# Lab 15-07 · Frequency-Domain Design: Bode, Nyquist and Margins

Four parts, each built on `freqdomain.py`. **Predict first**, then run.

In [1]:
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from freqdomain import freq_resp, margins, nyquist_encirclements, closed_loop_rhp, sensitivity_peak, bode_integral

## Part 1 · Bode and Nyquist with margins

*Predict:* which loop has a finite gain margin?

In [2]:
loops = {"100/[s(s+10)]": ([100], [1, 10, 0], 0.0),
         "8.30/[s(s+2)(s+4)]": ([224 / 27], [1, 6, 8, 0], 0.0),
         "0.5 e^(-s)/s": ([0.5], [1, 0], 1.0)}
w = np.logspace(-2, 2, 4000)
fig, axs = plt.subplots(2, 3, figsize=(12, 6))
for k, (name, (n, d, T)) in enumerate(loops.items()):
    L = freq_resp(n, d, w, T)
    m = margins(n, d, T)
    print(f"{name:20s} " + ", ".join(f"{key} = {val:.4g}" for key, val in m.items()))
    axs[0, k].semilogx(w, 20 * np.log10(np.abs(L))); axs[0, k].axhline(0, color="gray", lw=0.5)
    axs[0, k].set_title(name, fontsize=9); axs[0, k].set_ylabel("|L| (dB)")
    axs[1, k].plot(L.real, L.imag); axs[1, k].plot([-1], [0], "rx")
    axs[1, k].set_xlim(-3, 1); axs[1, k].set_ylim(-3, 1); axs[1, k].set_aspect("equal")
plt.tight_layout(); plt.show()

100/[s(s+10)]        wc = 7.862, pm = 51.83, delay_margin = 0.1151
8.30/[s(s+2)(s+4)]   wc = 0.9185, pm = 52.4, delay_margin = 0.9957, w180 = 2.828, gm = 5.786, gm_db = 15.25
0.5 e^(-s)/s         wc = 0.5, pm = 61.35, delay_margin = 2.142, w180 = 1.571, gm = 3.142, gm_db = 9.943


## Part 2 · Encirclements with an unstable plant

In [3]:
den = np.polymul([1, 0], np.polymul([1, -1], [1, 6]))
for K in (2, 5, 7, 8, 10, 30):
    Z = closed_loop_rhp([K, K], den)
    roots = np.roots(np.polyadd(den, [0, 0, K, K]))
    print(f"K = {K:3}: N = {nyquist_encirclements([K, K], den):+d}, Z = N + P = {Z}, unstable roots = {int(np.sum(roots.real > 0))}")

K =   2: N = +1, Z = N + P = 2, unstable roots = 2


K =   5: N = +1, Z = N + P = 2, unstable roots = 2


K =   7: N = +1, Z = N + P = 2, unstable roots = 2


K =   8: N = -1, Z = N + P = 0, unstable roots = 0


K =  10: N = -1, Z = N + P = 0, unstable roots = 0


K =  30: N = -1, Z = N + P = 0, unstable roots = 0


## Part 3 · The waterbed

In [4]:
w = np.logspace(-2, 2, 4000)
fig, ax = plt.subplots(figsize=(6, 3.2))
for K in (4, 224 / 27, 20):
    S = 1 / (1 + freq_resp([K], [1, 6, 8, 0], w))
    ax.semilogx(w, np.log(np.abs(S)), label=f"K = {K:.3g}")
    Ms, wm = sensitivity_peak([K], [1, 6, 8, 0])
    print(f"K = {K:5.3g}: Ms = {Ms:.3f} at {wm:.3f} rad/s, integral of ln|S| = {bode_integral([K], [1, 6, 8, 0]):.2e}")
ax.axhline(0, color="gray", lw=0.5); ax.set_xlabel("ω (rad/s)"); ax.set_ylabel("ln|S|"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

K =     4: Ms = 1.280 at 1.160 rad/s, integral of ln|S| = 5.03e-13
K =   8.3: Ms = 1.591 at 1.471 rad/s, integral of ln|S| = 1.90e-12
K =    20: Ms = 2.829 at 2.009 rad/s, integral of ln|S| = 8.85e-13


## Part 4 · Injection in ngspice

*Predict:* the phase margin with the second pole at 10 MHz.

In [5]:
def run(netlist_text):
    with open("_tmp_loop.cir", "w") as f:
        f.write(netlist_text)
    out = subprocess.run(["ngspice", "-b", "_tmp_loop.cir"], capture_output=True, text=True, timeout=120).stdout
    return [l for l in out.splitlines() if l.split("=")[0].strip() in ("fc", "pm")]
if shutil.which("ngspice"):
    base = open("opamp_loop.cir").read()
    print("f2 = 20 MHz:", run(base))
    print("f2 = 10 MHz:", run(base.replace("C2 x 0 7.95775n", "C2 x 0 15.9155n")))
    import os; os.remove("_tmp_loop.cir")
else:
    print("ngspice not installed: skip (CI runs opamp_loop.cir through tools/run_spice.py)")
A0, p1 = 1e5, 2 * math.pi * 100
for f2 in (20e6, 10e6):
    m = margins([A0], np.polymul([1 / p1, 1], [1 / (2 * math.pi * f2), 1]))
    print(f"Python, f2 = {f2 / 1e6:.0f} MHz: fc = {m['wc'] / 2 / math.pi / 1e6:.3f} MHz, PM = {m['pm']:.2f} deg")

f2 = 20 MHz: ['fc                  =  9.101820e+06', 'pm = 6.553080e+01']
f2 = 10 MHz: ['fc                  =  7.861525e+06', 'pm = 5.182800e+01']


Python, f2 = 20 MHz: fc = 9.102 MHz, PM = 65.53 deg
Python, f2 = 10 MHz: fc = 7.862 MHz, PM = 51.83 deg


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.